# CS224N L01 代码胶囊：负采样损失 vs 全 Softmax

> **这段代码在看什么**：对比 Word2Vec 训练中的两种目标函数——全 softmax（遍历整个词汇表）和负采样（只采样 k 个负例）。通过实际计算两种 loss 的值、梯度方向和计算量，理解为什么负采样能在保持梯度方向大致一致的同时实现数万倍加速。
>
> **和本讲哪个 waypoint 对应**：WP05（负采样目标函数 vs 全 softmax）
>
> **官方锚点**：Notes §3.5 (Eq.14-15, SGNS objective); R02 paper Section 3

In [1]:
import math
import json
import random

# ============================================================
# 1. 设置：小型玩具词汇表（纯标准库，Colab 无包运行）
# ============================================================

VOCAB = ["banking", "money", "crisis", "turning", "problems", "into"]
DIM = 3
SEED_V = 42
SEED_U = 99

def make_vectors(vocab, dim, seed):
    """确定性生成玩具向量，保证每次运行结果一致。"""
    vectors = {}
    for i, word in enumerate(vocab):
        vec = [round(math.sin(seed * (i + 1) * 7.3 + (d + 1) * 13.7) * 2.0, 4)
               for d in range(dim)]
        vectors[word] = vec
    return vectors

def dot(a, b):
    """向量点积。"""
    return sum(ai * bi for ai, bi in zip(a, b))

def sigmoid(x):
    """数值稳定的 sigmoid 函数。"""
    if x >= 0:
        return 1.0 / (1.0 + math.exp(-x))
    else:
        ez = math.exp(x)
        return ez / (1.0 + ez)

V = make_vectors(VOCAB, DIM, SEED_V)
U = make_vectors(VOCAB, DIM, SEED_U)
center = "banking"
output = "money"

print(f"词汇表: {VOCAB}")
print(f"|V| = {len(VOCAB)}, d = {DIM}")
print(f"中心词 c = '{center}', 输出词 o = '{output}'")
print()
print("Center vectors V:")
for w in VOCAB:
    print(f"  v_{w} = {V[w]}")
print("Context vectors U:")
for w in VOCAB:
    print(f"  u_{w} = {U[w]}")

词汇表: ['banking', 'money', 'crisis', 'turning', 'problems', 'into']
|V| = 6, d = 3
中心词 c = 'banking', 输出词 o = 'money'

Center vectors V:
  v_banking = [-0.2839, 1.6733, 1.7008]
  v_money = [-1.9769, -0.5621, 1.5009]
  v_crisis = [-0.8645, -1.9999, -0.8289]
  v_turning = [1.4746, -0.5997, -1.9824]
  v_problems = [1.7212, 1.6515, -0.3227]
  v_into = [-0.4748, 1.5591, 1.7949]
Context vectors U:
  u_banking = [1.9086, 1.3495, -0.7659]
  u_money = [1.9712, 1.1407, -1.0053]
  u_crisis = [1.9987, 0.9116, -1.2268]
  u_turning = [1.9905, 0.6662, -1.4264]
  u_problems = [1.9467, 0.4089, -1.6005]
  u_into = [1.8683, 0.1443, -1.7461]


## Part 1：全 Softmax 损失（Notes Eq.14）

> **这段代码在做什么**：计算中心词 "banking" 与所有 outside 词的点积，然后通过 softmax 得到概率分布。损失 = -log P(output|center)。
>
> **先看哪里**：P(money|banking) = 0.249037，损失 = 1.390153。需要计算 **6 个点积**（= |V|），当 |V|=100,000 时这就是瓶颈。

In [2]:
def softmax_loss(U, V, center, output, vocab):
    """
    全 softmax 交叉熵损失（Notes Eq.14）：
    J = -u_o^T v_c + log(sum_w exp(u_w^T v_c))
    需要遍历整个词汇表。"""
    vc = V[center]
    uo = U[output]
    scores = {w: dot(U[w], vc) for w in vocab}
    max_score = max(scores.values())
    log_sum_exp = max_score + math.log(sum(math.exp(s - max_score) for s in scores.values()))
    loss = -dot(uo, vc) + log_sum_exp
    total_exp = sum(math.exp(s) for s in scores.values())
    probs = {w: math.exp(scores[w]) / total_exp for w in vocab}
    # 梯度 dJ/dv_c
    grad_vc = [0.0] * DIM
    for d in range(DIM):
        for w in vocab:
            grad_vc[d] += probs[w] * U[w][d]
        grad_vc[d] -= uo[d]
    return loss, probs, grad_vc, len(vocab)

sm_loss, sm_probs, sm_grad, sm_dots = softmax_loss(U, V, center, output, VOCAB)

print(f"Dot products u_w^T v_{center}:")
for w in VOCAB:
    s = dot(U[w], V[center])
    marker = " <-- 输出词" if w == output else ""
    print(f"  u_{w}^T v_{center} = {s:.4f}{marker}")
print()

print(f"P(w|'{center}') under softmax:")
for w in VOCAB:
    bar = "#" * int(sm_probs[w] * 40)
    marker = " <-- 输出词" if w == output else ""
    print(f"  P({w:>10}|{center}) = {sm_probs[w]:.6f}  {bar}{marker}")
print()

print(f"Softmax loss = {sm_loss:.6f}")
print(f"  = -log(P('{output}'|'{center}')) = -log({sm_probs[output]:.6f}) = {-math.log(sm_probs[output]):.6f}")
print(f"需要的点积数: {sm_dots} (= |V|)")
print(f"Gradient dJ/dv_c = [{', '.join(f'{g:.4f}' for g in sm_grad)}]")

Dot products u_w^T v_banking:
  u_banking^T v_banking = 0.4136
  u_money^T v_banking = -0.3607 <-- 输出词
  u_crisis^T v_banking = -1.1286
  u_turning^T v_banking = -1.8764
  u_problems^T v_banking = -2.5906
  u_into^T v_banking = -3.2587

P(w|'banking') under softmax:
  P(   banking|banking) = 0.540196  #####################
  P(     money|banking) = 0.249037  ######### <-- 输出词
  P(    crisis|banking) = 0.115551  ####
  P(   turning|banking) = 0.054704  ##
  P(  problems|banking) = 0.026782  #
  P(      into|banking) = 0.013730  

Softmax loss = 1.390153
  = -log(P('money'|'banking')) = -log(0.249037) = 1.390153
需要的点积数: 6 (= |V|)
Gradient dJ/dv_c = [-0.0317, 0.0271, 0.0546]


## Part 2：负采样损失（Notes Eq.15 / R02 Section 3）

> **这段代码在做什么**：对同一个 (banking, money) 词对，用负采样目标计算损失。不再遍历整个词汇表，而是只计算正样本 + k 个随机负样本的 sigmoid。
>
> **先看哪里**：k=5 时只需要 6 个点积（= 1 + k），和全 softmax 的 6 个点积一样——因为 |V| 也只有 6。但当 |V|=100,000 时差异巨大。
>
> **容易误解的地方**：负采样损失值（1.702726）大于全 softmax（1.390153）——不是 bug。两个损失函数定义不同，绝对值不能直接比较。

In [3]:
def negative_sampling_loss(U, V, center, output, vocab, k, rng):
    """
    负采样损失（SGNS, Notes Eq.15）：
    J = -log(sigmoid(u_o^T v_c)) - sum_{i=1}^{k} log(sigmoid(-u_wi^T v_c))
    只需要 k+1 个点积。"""
    vc = V[center]
    uo = U[output]
    pos_score = dot(uo, vc)
    pos_loss = -math.log(sigmoid(pos_score) + 1e-10)
    candidates = [w for w in vocab if w != output]
    neg_words = [rng.choice(candidates) for _ in range(k)]
    neg_loss = 0.0
    neg_grad_vc = [0.0] * DIM
    for nw in neg_words:
        unw = U[nw]
        neg_score = dot(unw, vc)
        sig_neg = sigmoid(-neg_score)
        neg_loss += -math.log(sig_neg + 1e-10)
        sig_pos = sigmoid(neg_score)
        for d in range(DIM):
            neg_grad_vc[d] += sig_pos * unw[d]
    total_loss = pos_loss + neg_loss
    sig_pos_score = sigmoid(pos_score)
    grad_vc = [0.0] * DIM
    for d in range(DIM):
        grad_vc[d] = -(1.0 - sig_pos_score) * uo[d] + neg_grad_vc[d]
    return total_loss, neg_words, grad_vc, 1 + k, pos_loss, neg_loss

print("不同 k 值下的负采样损失：")
print()
for k in [2, 5, 10]:
    rng_k = random.Random(2024)
    ns_loss, neg_words, ns_grad, ns_dots, pos_l, neg_l = \
        negative_sampling_loss(U, V, center, output, VOCAB, k, rng_k)
    print(f"--- k = {k} ---")
    print(f"  正样本对: ({center}, {output})")
    print(f"  负样本: {neg_words}")
    print(f"  正样本 loss: -log(sigmoid(u_o^T v_c)) = {pos_l:.6f}")
    print(f"  负样本 loss: sum -log(sigmoid(-u_w^T v_c)) = {neg_l:.6f}")
    print(f"  总 NS loss = {ns_loss:.6f}")
    print(f"  需要的点积数: {ns_dots} (= 1 + k)")
    print()

不同 k 值下的负采样损失：

--- k = 2 ---
  正样本对: (banking, money)
  负样本: ['problems', 'crisis']
  正样本 loss: -log(sigmoid(u_o^T v_c)) = 0.889676
  负样本 loss: sum -log(sigmoid(-u_w^T v_c)) = 0.352569
  总 NS loss = 1.242245
  需要的点积数: 3 (= 1 + k)

--- k = 5 ---
  正样本对: (banking, money)
  负样本: ['problems', 'crisis', 'into', 'turning', 'crisis']
  正样本 loss: -log(sigmoid(u_o^T v_c)) = 0.889676
  负样本 loss: sum -log(sigmoid(-u_w^T v_c)) = 0.813050
  总 NS loss = 1.702726
  需要的点积数: 6 (= 1 + k)

--- k = 10 ---
  正样本对: (banking, money)
  负样本: ['problems', 'crisis', 'into', 'turning', 'crisis', 'problems', 'turning', 'into', 'crisis', 'problems']
  正样本 loss: -log(sigmoid(u_o^T v_c)) = 0.889676
  负样本 loss: sum -log(sigmoid(-u_w^T v_c)) = 1.418128
  总 NS loss = 2.307804
  需要的点积数: 11 (= 1 + k)



## Part 3：详细对比（k=5）

> **输出怎么解释**：
> - **损失值**：softmax=1.390153, NS=1.702726。NS 更大，但定义不同，不能直接比较。
> - **点积数**：都是 6（因为 |V|=6，k=5 时 1+k=6）。实际中 |V|=100,000 时 softmax 需要 100,000 个点积，NS 只需要 6 个。
> - **梯度余弦相似度**：-0.975359。负号表示梯度方向大致相反——但注意损失函数的梯度是「需要减去的量」，所以实际更新方向是一致的：都在拉近正样本、推远负样本。

In [4]:
rng5 = random.Random(2024)
ns_loss_5, neg_words_5, ns_grad_5, ns_dots_5, pos_l_5, neg_l_5 = \
    negative_sampling_loss(U, V, center, output, VOCAB, 5, rng5)

sm_grad_norm = math.sqrt(sum(g**2 for g in sm_grad))
ns_grad_norm = math.sqrt(sum(g**2 for g in ns_grad_5))
dot_grad = sum(a * b for a, b in zip(sm_grad, ns_grad_5))
grad_cos = dot_grad / (sm_grad_norm * ns_grad_norm) if sm_grad_norm > 0 and ns_grad_norm > 0 else 0.0

print(f"{'指标':<35} {'Softmax':>12} {'NS (k=5)':>12}")
print(f"{'-'*35} {'-'*12} {'-'*12}")
print(f"{'Loss':<35} {sm_loss:>12.6f} {ns_loss_5:>12.6f}")
print(f"{'Dot products':<35} {sm_dots:>12} {ns_dots_5:>12}")
print(f"{'Computation ratio (NS/Softmax)':<35} {'':>12} {ns_dots_5/sm_dots:>11.0%}")
print()
print(f"{'Gradient ||dJ/dv_c||':<35} {sm_grad_norm:>12.6f} {ns_grad_norm:>12.6f}")
print()
print(f"{'Gradient cosine similarity':<35} {grad_cos:>12.6f}")

指标                                       Softmax     NS (k=5)
----------------------------------- ------------ ------------
Loss                                    1.390153     1.702726
Dot products                                   6            6
Computation ratio (NS/Softmax)                          100%

Gradient ||dJ/dv_c||                    0.068661     0.480818

Gradient cosine similarity             -0.975359


## Part 4：k 值对损失和计算量的影响

> **先看哪里**：随着 k 增大，NS loss 单调上升（因为累加了更多负样本项），但每个负样本的贡献递减。梯度范数也随 k 增大。
>
> **关键观察**：k=5 是一个常用平衡点——足够提供稳定的梯度信号，又不会引入太多计算开销。

In [5]:
k_values = [1, 2, 5, 10, 20, 50]
print(f"{'k':>4}  {'loss':>10}  {'dots':>5}  {'grad_norm':>12}")
print("-" * 40)
for k in k_values:
    rng_k = random.Random(2024)
    ns_l, nw, ns_g, ns_d, pos_l_k, neg_l_k = \
        negative_sampling_loss(U, V, center, output, VOCAB, k, rng_k)
    ns_g_norm = math.sqrt(sum(g**2 for g in ns_g))
    print(f"{k:>4}  {ns_l:>10.6f}  {ns_d:>5}  {ns_g_norm:>12.6f}")

print()
print(f"Softmax loss (参考): {sm_loss:.6f}")
print(f"Softmax dots (参考): {sm_dots}")

   k        loss   dots     grad_norm
----------------------------------------
   1    0.961974      2      1.302805
   2    1.242245      3      0.705900
   5    1.702726      6      0.480818
  10    2.307804     11      1.815455
  20    4.159915     21      5.344550
  50   12.018401     51     20.493864

Softmax loss (参考): 1.390153
Softmax dots (参考): 6


## Part 5：为什么 Logistic 近似有效

> **这段代码在看什么**：负采样把多分类问题重新定义为二分类——「这个词对是真实的（context, center）对吗？」
>
> **输出怎么解释**：
> - 正样本：sigmoid(u_money^T v_banking) = 0.410789，模型需要把它推向 1.0
> - 负样本：sigmoid(u_problems^T v_banking) = 0.069747，已经接近 0——模型能区分它不是真实对
>
> 这就是 **噪声对比估计（NCE）** 的核心思想：不需要在整个词汇表上归一化，只需要学会区分真实对和 k 个噪声样本。

In [6]:
print("关键洞察：负采样把词预测重新定义为二分类")
print("  '这是真实的 (context, center) 词对吗？'")
print()
print("  正样本对 (c, o): 最大化 log(sigmoid(u_o^T v_c))")
print("    -> 让 sigmoid(u_o^T v_c) 趋近 1.0")
pos_sig = sigmoid(dot(U[output], V[center]))
print(f"    当前: sigmoid(u_{output}^T v_{center}) = sigmoid({dot(U[output], V[center]):.4f}) = {pos_sig:.6f}")
print()
print("  负样本对 (c, w_i): 最大化 log(sigmoid(-u_wi^T v_c))")
print("    -> 让 sigmoid(u_wi^T v_c) 趋近 0.0")
for nw in neg_words_5:
    neg_sig = sigmoid(dot(U[nw], V[center]))
    print(f"    sigmoid(u_{nw}^T v_{center}) = sigmoid({dot(U[nw], V[center]):.4f}) = {neg_sig:.6f}")
print()
print("这是噪声对比估计（NCE）近似：")
print("  不需要在整个 |V| 上归一化（昂贵），")
print("  只需要学会区分真实对和 k 个噪声样本。")

关键洞察：负采样把词预测重新定义为二分类
  '这是真实的 (context, center) 词对吗？'

  正样本对 (c, o): 最大化 log(sigmoid(u_o^T v_c))
    -> 让 sigmoid(u_o^T v_c) 趋近 1.0
    当前: sigmoid(u_money^T v_banking) = sigmoid(-0.3607) = 0.410789

  负样本对 (c, w_i): 最大化 log(sigmoid(-u_wi^T v_c))
    -> 让 sigmoid(u_wi^T v_c) 趋近 0.0
    sigmoid(u_problems^T v_banking) = sigmoid(-2.5906) = 0.069747
    sigmoid(u_crisis^T v_banking) = sigmoid(-1.1286) = 0.244421
    sigmoid(u_into^T v_banking) = sigmoid(-3.2587) = 0.037015
    sigmoid(u_turning^T v_banking) = sigmoid(-1.8764) = 0.132806
    sigmoid(u_crisis^T v_banking) = sigmoid(-1.1286) = 0.244421

这是噪声对比估计（NCE）近似：
  不需要在整个 |V| 上归一化（昂贵），
  只需要学会区分真实对和 k 个噪声样本。


## Part 6：真实世界加速比

> **先看哪个数字**：|V|=100,000, k=5 时，softmax 需要 100,000 个点积，负采样只需要 6 个——**16,667 倍加速**。
>
> **和课堂内容的对应**：这就是为什么实际 Word2Vec 训练都用负采样而不是全 softmax。当词汇表有几十万词时，全 softmax 每一步都要算几十万次点积，而负采样只需要算几个。

In [7]:
real_V_sizes = [1000, 10000, 50000, 100000]
k_for_speedup = 5
ns_dots_for_speedup = 1 + k_for_speedup

print(f"{'|V|':>10}  {'Softmax dots':>15}  {'NS dots (k=5)':>15}  {'Speedup':>10}")
print(f"{'-'*10}  {'-'*15}  {'-'*15}  {'-'*10}")
for vs in real_V_sizes:
    speedup = vs / ns_dots_for_speedup
    print(f"{vs:>10,}  {vs:>15,}  {ns_dots_for_speedup:>15}  {speedup:>9,.0f}x")

print()
print(f"|V|=100,000, k=5: softmax 需要 100,000 个点积, NS 只需要 {ns_dots_for_speedup} 个。")
print(f"每步训练加速 {100000 // ns_dots_for_speedup:,}x！")

       |V|     Softmax dots    NS dots (k=5)     Speedup
----------  ---------------  ---------------  ----------
     1,000            1,000                6        167x
    10,000           10,000                6      1,667x
    50,000           50,000                6      8,333x
   100,000          100,000                6     16,667x

|V|=100,000, k=5: softmax 需要 100,000 个点积, NS 只需要 6 个。
每步训练加速 16,666x！


## 总结

> **核心结论**：
> 1. 负采样用 k 个二分类替代 |V| 分类，计算量从 O(|V|) 降到 O(k)
> 2. 梯度方向大致一致（余弦相似度 -0.975）——都在「拉近正样本、推远负样本」
> 3. 当 |V|=100,000 时，这是 16,667 倍加速
> 4. 损失值不能直接比较——定义不同
>
> **容易误解的地方**：
> - 负采样不是近似 softmax——它是完全不同的二分类目标
> - σ(x) 是 sigmoid，不是 softmax
> - 本 capsule 用 toy 数据：|V|=6, d=3；实际用 |V|≈100k, d≈300